#환경 세팅

In [ ]:
!pip install "paramiko<3.0" sshtunnel --upgrade-strategy eager --quiet

In [ ]:
# !pip install sshtunnel psycopg2-binary

In [ ]:
from sshtunnel import SSHTunnelForwarder
import psycopg2
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import bigquery
from google.cloud import storage
import gspread
from google.auth import default
from google.auth.transport.requests import Request
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
import os
import gcsfs
import io
import paramiko
import json

#인증 정보

In [ ]:
# SSH 및 GCS 경로 설정
ssh_host = "YOUR_SSH_HOST_IP"
ssh_username = "YOUR_SSH_USERNAME"
gcs_pem_path = "gs://YOUR_SECURE_BUCKET/YOUR_KEY.pem"

# GCS에서 pem 파일 읽기 (문자열 디코딩)
fs = gcsfs.GCSFileSystem()
with fs.open(gcs_pem_path, "rb") as f:
    pem_file_content = f.read().decode("utf-8")

# paramiko 연동을 위해 PKey 객체로 변환
pkey = paramiko.RSAKey.from_private_key(io.StringIO(pem_file_content))


# RDS 접속 정보 설정
db_host = "YOUR_RDS_ENDPOINT.amazonaws.com"
db_port = 5432
db_user = "YOUR_DB_USER"
db_password = "YOUR_DB_PASSWORD"
db_name = "YOUR_DATABASE_NAME"

# 기존 구매자

In [ ]:
# SSH 터널링 및 DB 연결
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # 위에서 생성한 pkey 객체 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=("localhost", 5433),
) as tunnel:

    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password,
    )

    # 유저 데이터 조회
    df_oldUser = pd.read_sql(
        """
        SELECT
            uid,
            phone,
            user_login,
            user_name,
            TO_CHAR(regdate AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS signup_date
        FROM fivespot_user;
    """,
        conn,
    )
    conn.close()

df_oldUser.head()

In [ ]:
# SSH 터널링 및 DB 연결
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # pkey 객체 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=("localhost", 5433),
) as tunnel:

    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password,
    )

    # 과거 결제 내역 조회 (정상 결제 건만)
    df_oldPayment = pd.read_sql(
        """
      SELECT
            TO_CHAR(p.regdate AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS p_date,
            p.order_id,
            o.uid,
            (p.content::json) ->> 'name' AS name,
            p.total,
            c.contract_type,
            c.start_date,
            c.end_date
        FROM
            fivespot_payment p
        LEFT JOIN
            fivespot_order o ON p.order_id = o.order_id
        LEFT JOIN
            fivespot_contract c ON o.contract_id = c.contract_id
        WHERE
            p.status = 'buy'
            AND p.total > 0
            AND c.contract_type != 'service'
        ORDER BY
            p_date DESC;
                    """,
        conn,
    )
    conn.close()

df_oldPayment.head()

In [ ]:
df_oldPayment_sum = df_oldPayment.groupby('uid')['total'].sum().reset_index()
df_oldUser = df_oldUser.merge(df_oldPayment_sum, on='uid', how='left')
df_oldPurchaser = df_oldUser[df_oldUser['total'].notna()]
df_oldPurchaser['phone_number'] = df_oldPurchaser['phone'].str.replace('-', '', regex=False)
df_oldPurchaser_phone = df_oldPurchaser.groupby('phone_number')['total'].sum().reset_index()
df_oldPurchaser_phone['기존구매자'] = "기존구매자"
df_oldPurchaser_phone.head()

# User Raw

In [ ]:
# SSH 터널링 및 DB 연결
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # pkey 객체 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=("localhost", 5433),
) as tunnel:

    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password,
    )

    # 신규 가입 유저 정보 및 최신 약관 동의 여부 조회 (탈퇴 유저 제외)
    df_user = pd.read_sql(
        """
        SELECT
            TO_CHAR(u.created_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS signup_date,
            u.client_uid AS uid,
            u.name,
            u.phone_number,
            ui.email,
            ui.status,
            ua.is_agreed
        FROM client u
        LEFT JOIN client_info ui
            ON u.client_uid = ui.client_uid
        LEFT JOIN (
            SELECT ca.*
            FROM client_agreement ca
            INNER JOIN (
                SELECT client_uid, MAX(agreement_uid) AS max_agreement_uid
                FROM client_agreement
                GROUP BY client_uid
            ) latest
            ON ca.client_uid = latest.client_uid AND ca.agreement_uid = latest.max_agreement_uid
        ) ua
            ON u.client_uid = ua.client_uid
        WHERE ui.status != 'DELETED';
    """,
        conn,
    )
    conn.close()

df_user.head()

In [ ]:
# SSH 터널링 및 DB 연결
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # pkey 객체 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=("localhost", 5433),
) as tunnel:

    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password,
    )

    # 어드민(사내 유저) 계정 조회
    df_admin = pd.read_sql(
        """
        SELECT
            admin_uid,
            phone_number,
            email,
            name
        FROM admin;
    """,
        conn,
    )
    conn.close()

df_admin["admin"] = "admin"
df_admin.head()

In [ ]:
df_user = df_user.merge(df_admin[['phone_number', 'admin']], on='phone_number', how='left')
df_user = df_user.merge(df_oldPurchaser_phone[['phone_number', '기존구매자']], on='phone_number', how='left')
df_user = df_user[df_user['admin'].isna()]
df_user.head()

# 결제 Raw

In [ ]:
# SSH 터널링 및 DB 연결
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pkey,  # pkey 객체 사용
    remote_bind_address=(db_host, db_port),
    local_bind_address=("localhost", 5433),
) as tunnel:

    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password,
    )

    # 계약 및 결제 데이터 매핑 조회 (결제 완료 건만)
    df_payment = pd.read_sql(
        """
        SELECT
            TO_CHAR(c.created_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS c_date,
            c.contract_uid transaction_id,
            c.client_uid uid,
            c.status,
            c.product_name,
            pp.name product_period,
            TO_CHAR(c.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS start_date,
            TO_CHAR(c.initial_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS initial_end_date,
            TO_CHAR(c.actual_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS actual_end_date,
            ph.price,
            TO_CHAR(c.created_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS created_at,
            c.is_migrated,
            p.order_id
        FROM contract c
        LEFT JOIN payment p ON c.contract_uid = p.contract_payment_uid
        LEFT JOIN price_policy pp ON c.price_policy_uid = pp.price_policy_uid
        LEFT JOIN payment_history ph ON c.contract_uid = ph.payment_uid
        WHERE ph.payment_status = 'DONE'
        ORDER BY created_at ASC;
    """,
        conn,
    )
    conn.close()

df_payment.head()

In [ ]:
df_payment['product_name'] = df_payment['product_name'] + ' - ' + df_payment['product_period'].astype(str)
df_payment = df_payment.drop(columns=['product_period'])
df_payment.head()

In [ ]:
df_payment['row_num'] = df_payment.groupby('uid').cumcount() + 1
df_user_dedup = df_user.drop_duplicates(subset='uid', keep='first')
df_payment = df_payment.merge(df_user_dedup[['uid', '기존구매자']], on='uid', how='left')

# 구매 유형 분류 함수 정의
def classify_purchase(row):
    if (row['기존구매자'] == '기존구매자') or (row['row_num'] > 1):
        return '재구매'
    else:
        return '첫구매'

# 적용
df_payment['purchase_type'] = df_payment.apply(classify_purchase, axis=1)


In [ ]:
df_payment.head()

#빅쿼리 데이터와 DB 데이터 병합

In [ ]:
# 빅쿼리에서 GA4 웹 행동 로그 로드
client = bigquery.Client()
sql_query = """
    SELECT *
    FROM `fivespot-bigquery.MKT.MKT_eventAll_web` AS t1
    LEFT JOIN `fivespot-bigquery.MKT.MKT_userId_match_web` AS t2
    ON t1.user_pseudo_id = t2.user_pseudo_id
"""
df = client.query(sql_query).to_dataframe()
df = df.dropna(subset=["user_id"])

# 유저별 이벤트 타임스탬프 순서대로 시퀀스 체번
df["row_num2"] = (
    df.sort_values(by=["user_id", "event_timestamp"])
    .groupby("user_id")
    .cumcount()
    + 1
)
df = df.sort_values(by=["user_id", "row_num2"], ascending=[True, True])


# 결제 데이터와 유저 행동 로그 병합 (개편일 이후 기준)
df_payment["uid"] = df_payment["uid"].astype(str)
df_payment = df_payment[df_payment["c_date"] >= "2025-06-11"]
df_purchase = pd.merge(
    df_payment, df, left_on="uid", right_on="user_id", how="left"
)

# 휴먼에러로 잘못 들어간 utm 수정
df_purchase.loc[
    df_purchase['event_content'] == 'fs2147-250728-ua-megasale-7-lyj',
    ["event_source", "event_medium"],
] = ["mms", "paid"]

df_purchase.head()

In [ ]:
df_purchase["event_source"] = df_purchase["event_source"].astype(str)


# 유저 행동 로그 기반 마케팅 채널 분류 함수
def categorize_media(row):
    # 주요 DA 및 결제 매체
    if row["event_source"] in ["fbig", "fb", "ig"]:
        return "FBIG"
    elif row["event_source"] in ["tiktok"]:
        return "Tiktok"
    elif row["event_source"] in ["kakao"]:
        return "Kakao"
    elif row["event_source"] in ["toss"]:
        return "Toss"
    elif row["event_source"] in ["navergfa", "naver_gfa"]:
        return "NaverGFA"

    # 블로그 및 CRM 채널
    elif row["event_source"] and (
        "blog" in row["event_source"]
        or "brunch" in row["event_source"]
        or "tistory" in row["event_source"]
    ):
        return "Blog"
    elif row["event_source"] in ["mms", "alimtalk"]:
        return "CRM"

    # 검색광고(SA) 및 검색엔진(SEO) 조합 분기
    elif (
        row["event_source"]
        and "daum" in row["event_source"]
        and row["event_campaign"] == "sa"
    ):
        return "DaumSA"
    elif (
        row["event_source"]
        and "naver" in row["event_source"]
        and row["event_campaign"] in ["searchads", "sa"]
    ):
        return "NaverSA"
    elif (
        row["event_source"]
        and "naver" in row["event_source"]
        and row["event_campaign"] in ["brandsearch", "ba"]
    ):
        return "NaverBSA"
    elif (
        row["event_source"]
        and "daum" in row["event_source"]
        and row["event_medium"] == "organic"
    ):
        return "DaumSEO"
    elif row["event_source"] == "google" and row["event_medium"] == "organic":
        return "GoogleSEO"
    elif row["event_source"] == "google" and row["event_campaign"] in [
        "searchads",
        "19420764713",
        "17302820061",
    ]:
        return "GoogleSA"
    elif row["event_source"] == "google":
        return "GoogleDA"

    # 네이버 플레이스 및 기타 포털 자연유입
    elif (
        row["event_source"]
        and "place" in row["event_source"]
        and "kakao" not in row["event_source"]
    ):
        return "NaverPlace"
    elif row["event_source"] and "naver" in row["event_source"]:
        return "NaverSEO"

    # 자사 웹/앱 도메인 예외 처리
    elif row["event_source"] in ["fastfive.co.kr"]:
        return "fastfive.co.kr"
    elif row["event_source"] in ["app"]:
        return "ffapp"
    elif row["event_source"] in ["workanywhere.co.kr", "workanywhere"]:
        return "workanywhere.co.kr"
    elif row["event_source"] in ["oopy", "fivespot.oopy.io"]:
        return "oopy"
    else:
        return "ETC"


# 매핑 로직 반영
df_purchase["Media"] = df_purchase.apply(categorize_media, axis=1)

In [ ]:
# 1. 구매시간(created_at) → datetime으로 변환 후 KST timezone 지정
df_purchase['created_at_dt'] = pd.to_datetime(df_purchase['created_at'])
df_purchase['created_at_dt'] = df_purchase['created_at_dt'].dt.tz_localize('Asia/Seoul')

# (수정) 2. pd.to_datetime 호출 전, 유효하지 않은 타임스탬프 값을 미리 제거
# 타임스탬프는 일반적으로 양수이므로 음수 값을 가진 행을 먼저 제거합니다.
df_purchase = df_purchase[df_purchase['event_timestamp'] >= 0]

# 이제 안전하게 datetime으로 변환합니다.
df_purchase['event_time_dt'] = pd.to_datetime(df_purchase['event_timestamp'], unit='us', errors='coerce')

# NaT 값 행 제거 (혹시 모를 다른 변환 오류 값 처리)
df_purchase.dropna(subset=['event_time_dt'], inplace=True)

# 시간대 변환
df_purchase['event_time_dt'] = df_purchase['event_time_dt'].dt.tz_localize('UTC')
df_purchase['event_time_kst'] = df_purchase['event_time_dt'].dt.tz_convert('Asia/Seoul')

# 3. 방문 시간(event_time_kst)이 구매 시간(created_at_dt)보다 늦은 경우 제거
df_purchase = df_purchase[df_purchase['event_time_kst'] <= df_purchase['created_at_dt']]

In [ ]:
# 분석에서 제외할 내부망, PG사, 이탈 인증 도메인 리스트
exclude_sources = [
    "ffspot.co.kr",
    "logins.daum.net",
    "kauth.kakao.com",
    "fivespot.channel.io",
    "accounts.kakao.com",
    "fivespot.oopy.io",
    "p745j.channel.io",
    "ksmobile.inicis.com",
    "form.jotform.com",
    "payment-gateway.tosspayments.com",
    "google-play",
    "payment-widget.tosspayments.com",
]

# 노이즈 도메인 제외 및 마이그레이션 데이터 필터링
df_purchase = df_purchase[~df_purchase["event_source"].isin(exclude_sources)]
df_purchase = df_purchase[~df_purchase["is_migrated"]]

df_purchase.head()

### Lookback window

In [ ]:
# ConvDuration 컬럼 생성 (일 단위 차이)
df_purchase['ConvDuration'] = (df_purchase['created_at_dt'] - df_purchase['event_time_kst']).dt.days
df_purchase = df_purchase[(df_purchase['ConvDuration'] >= 0) & (df_purchase['ConvDuration'] < 8)]


# 다채널 Data Driven

In [ ]:


# 1. row_num2 숫자 변환
df_purchase['row_num2'] = pd.to_numeric(df_purchase['row_num2'], errors='coerce')

# 2. uid별 최대 row_num2 구하기
max_row_num2 = df_purchase.groupby('uid')['row_num2'].transform('max')

# 3. 최대값 행만 필터링
df_LastClick = df_purchase[df_purchase['row_num2'] == max_row_num2].copy()
df_LastClick.head()

In [ ]:
df_LastClick = df_LastClick.drop(columns=[
    'created_at',
    'is_migrated',
    'row_num_x',
    'row_num_y',
    'event_timestamp',
    'traffic_type',
    'user_pseudo_id_1',
    'user_id',
    'event_time_dt'
])


In [ ]:


# 2. uid별 최소 row_num2 구하기
min_row_num2 = df_purchase.groupby('uid')['row_num2'].transform('min')

# 3. 최소값 행만 필터링
df_1stClick = df_purchase[df_purchase['row_num2'] == min_row_num2].copy()

df_1stClick.head()


In [ ]:
df_1stClick = df_1stClick.drop(columns=[
    'created_at',
    'is_migrated',
    'row_num_x',
    'row_num_y',
    'event_timestamp',
    'traffic_type',
    'user_pseudo_id_1',
    'user_id',
    'event_time_dt'
])


# 결과 전송

##대시보드 전송

In [ ]:
df_LastClick = df_LastClick.applymap(str)
df_LastClick["price"] = (
    pd.to_numeric(df_LastClick["price"], errors="coerce").astype("Int64")
)

# GCS에서 서비스 계정 키 로드
BUCKET_NAME = "YOUR_BUCKET_NAME"
KEY_FILE_IN_BUCKET = "YOUR_SERVICE_ACCOUNT_KEY.json"

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# 구글 시트 API 인증
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# 대상 시트 및 워크시트 오픈
sheet_id = "YOUR_SPREADSHEET_ID"
worksheet = gc.open_by_key(sheet_id).worksheet("U_LastClick")

# 결측치 빈 문자열 처리 및 헤더 포함 리스트 변환
df_for_upload = df_LastClick.fillna("")
data_to_upload = [
    df_for_upload.columns.values.tolist()
] + df_for_upload.values.tolist()

# A1 셀부터 데이터 덮어쓰기
worksheet.update("A1", data_to_upload)

print("스프레드시트 업데이트 완료")

In [ ]:
df_1stClick = df_1stClick.applymap(str)
df_1stClick["price"] = (
    pd.to_numeric(df_1stClick["price"], errors="coerce").astype("Int64")
)

# GCS에서 서비스 계정 키 로드
BUCKET_NAME = "YOUR_BUCKET_NAME"
KEY_FILE_IN_BUCKET = "YOUR_SERVICE_ACCOUNT_KEY.json"

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# 구글 시트 API 인증
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# 대상 시트 및 워크시트 오픈
sheet_id = "YOUR_SPREADSHEET_ID"
worksheet = gc.open_by_key(sheet_id).worksheet("U_1stClick")

# 결측치 빈 문자열 처리 및 헤더 포함 리스트 변환
df_for_upload = df_1stClick.fillna("")
data_to_upload = [
    df_for_upload.columns.values.tolist()
] + df_for_upload.values.tolist()

# A1 셀부터 데이터 덮어쓰기
worksheet.update("A1", data_to_upload)

print("스프레드시트 업데이트 완료")

# 구글시트에서 광고데이터 가져오기

In [ ]:
# GCS에서 서비스 계정 키 로드 및 gspread 인증
BUCKET_NAME = "YOUR_BUCKET_NAME"
KEY_FILE_IN_BUCKET = "YOUR_SERVICE_ACCOUNT_KEY.json"

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

gc = gspread.service_account_from_dict(key_file_dict)


# 구글 시트 데이터 로드 함수
def get_spreadsheet_data(url, sheet_name):
    try:
        spreadsheet = gc.open_by_url(url)
        sheet = spreadsheet.worksheet(sheet_name)
        data = sheet.get_all_records()
        return pd.DataFrame(data)
    except gspread.SpreadsheetNotFound:
        print(f"시트를 찾을 수 없습니다: {url}")
        raise
    except gspread.exceptions.APIError as e:
        print(f"API 권한 오류 발생: {e}")
        raise


# 수치형 컬럼 변환 헬퍼 함수
def convert_to_integer(df, columns):
    for column in columns:
        df[column] = (
            pd.to_numeric(df[column], errors="coerce").fillna(0).astype(int)
        )
    return df


# 메타 광고 데이터 로드 및 전처리
url2 = "https://docs.google.com/spreadsheets/d/YOUR_META_SPREADSHEET_ID/"
df_metaRaw = get_spreadsheet_data(url2, "meta raw")

# 날짜 누락 건 필터링 및 공백 데이터 0 처리
df_metaRaw = df_metaRaw[
    ~((pd.isnull(df_metaRaw["Date"])) | (df_metaRaw["Date"].str.strip() == ""))
]
df_metaRaw = df_metaRaw.replace(r"^\s*$", 0, regex=True)

# 텍스트 타입 캐스팅 및 파라미터 기반 utm_content 파싱
df_metaRaw["Date"] = df_metaRaw["Date"].astype(str)
df_metaRaw["Ad url tags"] = df_metaRaw["Ad url tags"].astype(str)
df_metaRaw["Destination URL"] = df_metaRaw["Destination URL"].astype(str)
df_metaRaw["Cost"] = df_metaRaw["Cost"].astype(str)
df_metaRaw["Website purchases conversion value"] = df_metaRaw[
    "Website purchases conversion value"
].astype(str)
df_metaRaw["utm_content"] = df_metaRaw["Ad url tags"].str.extract(
    r"utm_content=([^&]+)"
)

df_metaRaw.tail()

In [ ]:
# 구글 광고 raw 데이터 로드
url3 = "https://docs.google.com/spreadsheets/d/YOUR_GOOGLE_SPREADSHEET_ID/"
df_GoogleRaw = get_spreadsheet_data(url3, "google raw")
df_GoogleRaw.head()

In [ ]:
# 카카오 광고 raw 데이터 로드
url4 = "https://docs.google.com/spreadsheets/d/YOUR_KAKAO_GFA_SPREADSHEET_ID/"
df_KakaoRaw = get_spreadsheet_data(url4, "kakao raw")
df_KakaoRaw.head()

In [ ]:
# 네이버 GFA 광고 raw 데이터 로드
url5 = "https://docs.google.com/spreadsheets/d/YOUR_KAKAO_GFA_SPREADSHEET_ID/"
df_gfaRaw = get_spreadsheet_data(url5, "gfa raw")
df_gfaRaw.head()

In [ ]:
# BigQuery 명명 규칙에 맞게 컬럼명 정제
def rename_columns(df):
    return df.rename(
        columns=lambda x: x.strip()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("-", "_")
    )


df_metaRaw = rename_columns(df_metaRaw)

# 스키마 불일치 방지를 위해 텍스트 컬럼 cast 및 결측치 처리
cols_to_fix = [
    "Campaign_name",
    "Ad_set_name",
    "Ad_name",
    "utm_content",
    "Ad_url_tags",
    "Destination_URL",
]

for col in cols_to_fix:
    if col in df_metaRaw.columns:
        df_metaRaw[col] = df_metaRaw[col].astype(str)
        df_metaRaw[col] = df_metaRaw[col].replace("nan", None)

print("\n--- 데이터 타입 변환 및 컬럼명 정제 완료 ---")
print(df_metaRaw.info())


# BigQuery 클라이언트 초기화 및 데이터 적재 (Overwrite)
bq_client = bigquery.Client.from_service_account_info(
    key_file_dict, project="YOUR_PROJECT_ID"
)

dataset_id = "MKT"
table_id = "meta_raw"
table_ref = bq_client.dataset(dataset_id).table(table_id)

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

job = bq_client.load_table_from_dataframe(
    df_metaRaw, table_ref, job_config=job_config
)
job.result()

print(f"\n{job.output_rows} rows uploaded to {dataset_id}.{table_id} successfully.")

In [ ]:
print("\n--- BigQuery에서 메타 데이터 쿼리 시작 ---")

# 과거 히스토리(meta_fix)와 신규 수집 데이터(meta_raw) 통합
sql_query = """
SELECT * FROM `your-project-id.MKT.meta_fix`
UNION ALL
SELECT * FROM `your-project-id.MKT.meta_raw`
ORDER BY Date
"""

df_meta = bq_client.query(sql_query).to_dataframe()

# utm_content 누락 건 필터링
df_meta = df_meta[df_meta["utm_content"].notna() & (df_meta["utm_content"] != "")]

# 수치형 데이터 타입 변환
df_meta["Cost"] = pd.to_numeric(df_meta["Cost"], errors="coerce")
df_meta["Website_purchases_conversion_value"] = pd.to_numeric(
    df_meta["Website_purchases_conversion_value"], errors="coerce"
)

# 소재 매핑용 마스터 인덱스 생성
meta_index = (
    df_meta[["Campaign_name", "Ad_set_name", "utm_content"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# 일별/소재별 실적 집계
df_meta_grouped = df_meta.groupby(
    ["Date", "Campaign_name", "Ad_set_name", "utm_content"], as_index=False
).agg({"Cost": "sum", "Website_purchases_conversion_value": "sum"})

df_meta_grouped.rename(
    columns={"Website_purchases_conversion_value": "매체거래액"}, inplace=True
)

# 날짜 포맷 변환 및 분석 기준일 이후 데이터만 필터링
df_meta_grouped["Date"] = pd.to_datetime(
    df_meta_grouped["Date"], errors="coerce"
)
df_meta_grouped = df_meta_grouped[
    df_meta_grouped["Date"] >= pd.to_datetime("2025-06-16")
]

print("\n--- 최종 데이터 처리 완료 ---")
df_meta_grouped.head()

In [ ]:
df_GoogleRaw.rename(columns={'Campaign name': 'Campaign_name'}, inplace=True)
df_GoogleRaw.rename(columns={'Ad group name': 'Ad_set_name'}, inplace=True)
df_GoogleRaw.rename(columns={'Ad ID': 'utm_content'}, inplace=True)
df_GoogleRaw.rename(columns={'Total conversion value': '매체거래액'}, inplace=True)

df_GoogleRaw.head()

In [ ]:
df_GoogleRaw['Date'] = pd.to_datetime(df_GoogleRaw['Date'], errors='coerce')
df_GoogleRaw['utm_content'] = df_GoogleRaw['utm_content'].astype(str)
df_GoogleRaw['Cost'] = df_GoogleRaw['Cost'].astype(int)
df_GoogleRaw = df_GoogleRaw[df_GoogleRaw['Date'] >= pd.to_datetime('2025-06-16')]


In [ ]:
df_KakaoRaw.head()

In [ ]:
df_KakaoRaw.rename(columns={'날짜': 'Date'}, inplace=True)
df_KakaoRaw.rename(columns={'캠페인명': 'Campaign_name'}, inplace=True)
df_KakaoRaw.rename(columns={'광고그룹명': 'Ad_set_name'}, inplace=True)
df_KakaoRaw.rename(columns={'소재명': 'utm_content'}, inplace=True)
df_KakaoRaw.rename(columns={'비용': 'Cost'}, inplace=True)
df_KakaoRaw['매체거래액'] = 0
df_KakaoRaw.head()

In [ ]:
# 카카오 소재명 해상도 접미사 제거 (_숫자x숫자 패턴 정형화)
df_KakaoRaw["utm_content"] = df_KakaoRaw["utm_content"].str.replace(
    r"_\d+x\d+", "", regex=True
)

# 일별/소재별 실적 집계
df_KakaoRaw_2 = df_KakaoRaw.groupby(
    ["Date", "Campaign_name", "Ad_set_name", "utm_content"], as_index=False
)[["Cost", "매체거래액"]].sum()

df_KakaoRaw_2.head()

In [ ]:
# Date 컬럼을 datetime 타입으로 변경
df_KakaoRaw_2['Date'] = pd.to_datetime(df_KakaoRaw_2['Date'])

# 매체거래액 컬럼을 float 타입으로 변경
df_KakaoRaw_2['매체거래액'] = df_KakaoRaw_2['매체거래액'].astype(float)

In [ ]:
df_gfaRaw.head()

In [ ]:
df_gfaRaw.rename(columns={'날짜': 'Date'}, inplace=True)
df_gfaRaw.rename(columns={'캠페인': 'Campaign_name'}, inplace=True)
df_gfaRaw.rename(columns={'광고그룹': 'Ad_set_name'}, inplace=True)
df_gfaRaw.rename(columns={'광고소재': 'utm_content'}, inplace=True)
df_gfaRaw.rename(columns={'총비용': 'Cost'}, inplace=True)
df_gfaRaw.rename(columns={'총전환매출액': '매체거래액'}, inplace=True)
df_gfaRaw = df_gfaRaw.drop(columns=['총전환수'])
df_gfaRaw.head()

In [ ]:
# Date 컬럼을 datetime 타입으로 변경
df_gfaRaw['Date'] = pd.to_datetime(df_gfaRaw['Date'])

# 매체거래액 컬럼을 float 타입으로 변경
df_gfaRaw['매체거래액'] = df_gfaRaw['매체거래액'].astype(float)

In [ ]:
df_gfaRaw.info()

In [ ]:
df_gfaRaw = df_gfaRaw.groupby(
    ['Date', 'Campaign_name', 'Ad_set_name', 'utm_content'],
    as_index=False
)[['Cost', '매체거래액']].sum()

In [ ]:
df_DA_MediaRaw = pd.concat([df_meta_grouped, df_GoogleRaw, df_KakaoRaw_2, df_gfaRaw], ignore_index=True).drop_duplicates()
df_DA_MediaRaw.head()

# DA only dataDriven

In [ ]:
import pandas as pd

# 1. 기본 필터링 (매체 선택 및 첫구매 기준)
df_DA = df_purchase[df_purchase['Media'].isin(['FBIG', 'GoogleDA', 'Kakao', 'NaverGFA'])].copy()
df_DA = df_DA[df_DA['purchase_type'] == '첫구매']

# 2. row_num2 숫자 변환 (순서 파악 용도)
df_DA['row_num2'] = pd.to_numeric(df_DA['row_num2'], errors='coerce')

# ---------------------------------------------------------
# [모델 1] Last Click (마지막 클릭)
# ---------------------------------------------------------
max_row_num2 = df_DA.groupby('uid')['row_num2'].transform('max')
df_LastClick = df_DA[df_DA['row_num2'] == max_row_num2].copy()

df_LastClick_grouped = df_LastClick.groupby(['c_date', 'event_content'])[['uid', 'price']].agg({'uid': 'nunique', 'price': 'sum'})
df_LastClick_grouped = df_LastClick_grouped.rename(columns={'uid': '구매수LastClick', 'price': '거래액LastClick'}).reset_index()

# ---------------------------------------------------------
# [모델 2] 1st Click (첫 클릭 / 기존 코드의 7days)
# ---------------------------------------------------------
min_row_num2 = df_DA.groupby('uid')['row_num2'].transform('min')
df_1stClick = df_DA[df_DA['row_num2'] == min_row_num2].copy()

df_1stClick_grouped = df_1stClick.groupby(['c_date', 'event_content'])[['uid', 'price']].agg({'uid': 'nunique', 'price': 'sum'})
df_1stClick_grouped = df_1stClick_grouped.rename(columns={'uid': '구매수7days', 'price': '거래액7days'}).reset_index()

# ---------------------------------------------------------
# [모델 3] Linear Click (선형 기여 - 신규 추가)
# ---------------------------------------------------------
# 조건: 동일소재 두번 본 것은 1번만 카운팅
df_linear_base = df_DA.drop_duplicates(subset=['uid', 'event_content']).copy()

# 각 uid별로 기여도 분모(n) 구하기 (한 유저가 본 고유 소재의 개수)
df_linear_base['n'] = df_linear_base.groupby('uid')['event_content'].transform('count')

# 기여도 계산 (1건을 n으로 나누고, 거래액을 n으로 나눔)
df_linear_base['구매수Linear'] = 1 / df_linear_base['n']
df_linear_base['거래액Linear'] = df_linear_base['price'] / df_linear_base['n']

# 소재별 합계 집계
df_Linear_grouped = df_linear_base.groupby(['c_date', 'event_content'])[['구매수Linear', '거래액Linear']].sum().reset_index()

# ---------------------------------------------------------
# 데이터 병합 (LastClick + 1stClick + Linear)
# ---------------------------------------------------------
# 1. 모델간 병합
merged_df_DA = pd.merge(
    df_LastClick_grouped,
    df_1stClick_grouped,
    how='outer',
    on=['c_date', 'event_content']
)

merged_df_DA = pd.merge(
    merged_df_DA,
    df_Linear_grouped,
    how='outer',
    on=['c_date', 'event_content']
)

# 2. 메타 정보(Campaign, Ad_set 등) 인덱스 생성 및 병합
df_DA_MediaRaw_index = df_DA_MediaRaw[['utm_content', 'Campaign_name', 'Ad_set_name']].drop_duplicates(subset='utm_content', keep='first')

merged_df_DA = merged_df_DA.merge(
    df_DA_MediaRaw_index,
    how='left',
    left_on='event_content',
    right_on='utm_content'
)

# 3. 데이터 정제
merged_df_DA.fillna(0, inplace=True)
merged_df_DA = merged_df_DA[merged_df_DA['utm_content'] != 0]
merged_df_DA['c_date'] = pd.to_datetime(merged_df_DA['c_date'], errors='coerce')

# 4. 원본 MediaRaw 데이터와 최종 병합
merged_df = df_DA_MediaRaw.merge(
    merged_df_DA,
    how='outer',
    left_on=['Date', 'Campaign_name', 'Ad_set_name', 'utm_content'],
    right_on=['c_date', 'Campaign_name', 'Ad_set_name', 'utm_content']
)

# 5. 후처리 (날짜 채우기 및 불필요 컬럼 제거)
merged_df['Date'].fillna(merged_df['c_date'], inplace=True)
merged_df.fillna(0, inplace=True)
merged_df.drop(['c_date', 'event_content'], axis=1, errors='ignore', inplace=True)

# 최종 결과 확인
merged_df.head()

In [ ]:
# 숫자형 컬럼들 중에서 float이지만 정수로 표현 가능한 경우 int로 변환
for col in merged_df.select_dtypes(include='number').columns:
    # NaN 값 있는 경우 int 변환이 불가능하므로 우선 채움 또는 스킵 필요
    if merged_df[col].isnull().any():
        continue  # 또는: merged_df[col] = merged_df[col].fillna(0)

    # 모든 값이 정수처럼 보이면 int로 변환
    if (merged_df[col] % 1 == 0).all():
        merged_df[col] = merged_df[col].astype(int)
merged_df.head()

## 대시보드 전송

In [ ]:
# GCS에서 서비스 계정 키 로드 및 gspread 인증
BUCKET_NAME = "YOUR_BUCKET_NAME"
KEY_FILE_IN_BUCKET = "YOUR_SERVICE_ACCOUNT_KEY.json"

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

gc = gspread.service_account_from_dict(key_file_dict)

# 대상 시트 및 워크시트 오픈
sheet_id = "YOUR_SPREADSHEET_ID"
worksheet = gc.open_by_key(sheet_id).worksheet("U_tableau raw")


# 구글 시트 업로드용 전처리
print("\n--- Google Sheets 업로드 시작 ---")
df_for_upload = merged_df.copy()

# datetime 타입 스트링 변환 (gspread 업로드 에러 방지)
if "Date" in df_for_upload.columns and pd.api.types.is_datetime64_any_dtype(
    df_for_upload["Date"]
):
    df_for_upload["Date"] = df_for_upload["Date"].dt.strftime("%Y-%m-%d")

# 결측치 빈 문자열 처리
df_for_upload = df_for_upload.fillna("")


# 데이터 업로드 (A1 셀부터 덮어쓰기)
data_to_upload = [
    df_for_upload.columns.values.tolist()
] + df_for_upload.values.tolist()
worksheet.update(range_name="A1", values=data_to_upload)

print("스프레드시트 업데이트 완료")

##빅쿼리 최종데이터 업로드

In [ ]:
# GCS에서 서비스 계정 키 로드
BUCKET_NAME = "YOUR_BUCKET_NAME"
KEY_FILE_IN_BUCKET = "YOUR_SERVICE_ACCOUNT_KEY.json"

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())


# BigQuery 업로드용 데이터 전처리 (타입 및 포맷 통일)
merged_df = merged_df.applymap(str)
merged_df["Date"] = pd.to_datetime(merged_df["Date"]).dt.strftime("%Y-%m-%d")


# BigQuery 규칙에 맞게 컬럼명 정제
def rename_columns(df):
    return df.rename(
        columns=lambda x: x.strip()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("-", "_")
    )


merged_df = rename_columns(merged_df)


# BigQuery 클라이언트 초기화 및 데이터 적재
print("\n--- BigQuery 업로드 시작 ---")
bq_client = bigquery.Client.from_service_account_info(
    key_file_dict, project="YOUR_PROJECT_ID"
)

dataset_id = "MKT"
table_id = "U_DA_raw"
table_ref = bq_client.dataset(dataset_id).table(table_id)

# Overwrite(덮어쓰기) 설정
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# 메모리 관리 및 안정적인 적재를 위해 5만 건씩 청크 단위로 분할 업로드
chunk_size = 50000
for i in range(0, len(merged_df), chunk_size):
    chunk = merged_df.iloc[i : i + chunk_size]
    job = bq_client.load_table_from_dataframe(
        chunk, table_ref, job_config=job_config
    )
    job.result()  # 각 청크 적재 완료 대기
    print(f"Uploaded chunk {i // chunk_size + 1}.")

print(
    f"\n모든 데이터를 {dataset_id}.{table_id} 테이블에 성공적으로 업로드했습니다."
)